In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import sys
from pathlib import Path

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import chronopack as cp
from chronopack.tl import _gene_dynamics as gd
from chronopack.preprocessing import filter_low_count_genes, filter_technical_genes


In [2]:
adata_hvg = sc.read_h5ad('../output/adata_hvg.chk1.h5ad')
adata = sc.read_h5ad('../output/adata_full.chk1.h5ad')

# Compute CPM & logCPM matrices for negative bulk control

In [3]:
mtx_agg, cpm_ref, logcpm_ref = gd.load_negative_control_logcpm('../intmd/k562_neg-ctrl_mtx-agg.pkl')


# Compute pseudobulk CPM & logCPM matrices for LPS+THP1 experiment

In [4]:
cpm, logcpm = gd.temporal_pseudobulk_from_anndata(adata)


# Prioritize genes by variation over time

In [5]:
def RenameClusters(x):
    if (x == '0'): return 'K562'
    if (x == '1'): return 'K562'
    if (x == '2'): return 'THP1'

adata.obs['cell_type'] = adata.obs['traj_clusters'].apply(RenameClusters)

# New approach

In [6]:
adata_thp1_hvg = adata[(adata.obs['cell_type'] == 'THP1'), :].copy()
adata_k562_hvg = adata[(adata.obs['cell_type'] == 'K562'), :].copy()

In [7]:
cpm_thp1, logcpm_thp1 = gd.temporal_pseudobulk_from_anndata(adata_thp1_hvg)
cpm_k562, logcpm_k562 = gd.temporal_pseudobulk_from_anndata(adata_k562_hvg)

In [8]:
adata_k562_hvg.var["total_counts"] = np.asarray(
    adata_k562_hvg.layers["counts"].sum(axis=0)
).ravel()

In [9]:
adata_thp1_hvg.var["total_counts"] = np.asarray(
    adata_thp1_hvg.layers["counts"].sum(axis=0)
).ravel()

In [10]:
low_count_genes_k562 = set(adata_k562_hvg.var.index[adata_k562_hvg.var["total_counts"] < 10])
low_count_genes_thp1 = set(adata_thp1_hvg.var.index[adata_thp1_hvg.var["total_counts"] < 10])
print(f"K562 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_k562)}")
print(f"THP1 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_thp1)}")


K562 low-count genes to remove before tradeSeq input writing: 7253
THP1 low-count genes to remove before tradeSeq input writing: 10695


In [11]:
adata_hvg = adata.copy()
shared_genes = list(set(adata_hvg.var.index) & set(logcpm_ref.columns) & set(logcpm.columns))

cpm_ref = cpm_ref.loc[:, shared_genes]
logcpm_ref = logcpm_ref.loc[:, shared_genes]

cpm = cpm.loc[:, shared_genes]
logcpm = logcpm.loc[:, shared_genes]
logcpm.shape

(12, 19433)

# Compute correlations

In [12]:
corr_k562 = gd.compute_smooth_correlations(
    logcpm_k562,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)

corr_thp1 = gd.compute_smooth_correlations(
    logcpm_thp1,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.7,
    local_fraction_threshold=0.3,
)


In [13]:
corr_k562 = gd.refilter_tag_bias_candidates(
    corr_k562,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_k562["tag_bias_candidate"].sum()))

corr_k562[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


Tag-bias candidates: 857


                 pearson_r_residual  ...  tag_bias_candidate
gene                                 ...                    
ENSG00000279881                 1.0  ...                True
GRAMD1B                         1.0  ...                True
ENSG00000302483                 1.0  ...                True
DDX25                           1.0  ...                True
LINC02029                       1.0  ...                True
ENSG00000297147                 1.0  ...                True
LRRC34                          1.0  ...                True
ENSG00000285954                 1.0  ...                True
COL4A1                          1.0  ...                True
ENSG00000279118                 1.0  ...                True
BOLA3P3                         1.0  ...                True
RBMS2P1                         1.0  ...                True
MAP6                            1.0  ...                True
NPR3                            1.0  ...                True
KCNG1                   

In [14]:
corr_thp1 = gd.refilter_tag_bias_candidates(
    corr_thp1,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_thp1["tag_bias_candidate"].sum()))

corr_thp1[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


Tag-bias candidates: 507


                 pearson_r_residual  ...  tag_bias_candidate
gene                                 ...                    
ENSG00000287159                 1.0  ...                True
SRSF9P1                         1.0  ...                True
ENSG00000304531                 1.0  ...                True
SMARCE1P3                       1.0  ...                True
NUDT9P1                         1.0  ...                True
DDX25                           1.0  ...                True
ENSG00000295329                 1.0  ...                True
AP4B1-AS1                       1.0  ...                True
NPR3                            1.0  ...                True
ENSG00000303873                 1.0  ...                True
P4HA3                           1.0  ...                True
EML4-AS1                        1.0  ...                True
GALNT4                          1.0  ...                True
ENSG00000230186                 1.0  ...                True
ENSG00000260342         

In [15]:
excl_genes_k562 = gd.tag_bias_exclusion_genes(corr_k562)
print(len(excl_genes_k562))
print('HNRNPA1P27' in excl_genes_k562)

857
True


In [16]:
excl_genes_thp1 = gd.tag_bias_exclusion_genes(corr_thp1)
print(len(excl_genes_thp1))
print('HNRNPA1P27' in excl_genes_thp1)

507
True


# Pre-filter @K562

In [17]:
adata_k562_hvg = filter_technical_genes(adata_k562_hvg, exclude_genes=excl_genes_k562)
adata_k562_hvg = filter_low_count_genes(adata_k562_hvg, min_total_counts=10, counts_layer="counts")
adata_k562_hvg


AnnData object with n_obs × n_vars = 3128 × 11928
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden_unintegrated', 'THP1_score', 'K562_score', 'THP1_minus_K562', 'cell_type', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'chronopack', 'hvg', 'leiden_unintegrated', 'leiden_unintegrated_colors', 'log1p', 'neighbors', 'ot_graph', 'pca', 'phase_colors', 'sample_colors', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj_fates', '

In [18]:
print('PLAU' in set(adata_k562_hvg.var.index)) # similar shape
print('CD83' in set(adata_k562_hvg.var.index)) # similar shape
print('EGR1' in set(adata_k562_hvg.var.index)) # similar shape
print('IER3' in set(adata_k562_hvg.var.index)) # similar shape

True
True
True
True


In [19]:
# TPs
print('REL' in set(adata_k562_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_k562_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_k562_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_k562_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_k562_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_k562_hvg.var.index)) # similar shape

True
True
True
True
True
True


# Pre-filter @THP1

In [20]:
adata_thp1_hvg = filter_technical_genes(adata_thp1_hvg, exclude_genes=excl_genes_thp1)
adata_thp1_hvg = filter_low_count_genes(adata_thp1_hvg, min_total_counts=10, counts_layer="counts")
adata_thp1_hvg


AnnData object with n_obs × n_vars = 787 × 10117
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden_unintegrated', 'THP1_score', 'K562_score', 'THP1_minus_K562', 'cell_type', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'chronopack', 'hvg', 'leiden_unintegrated', 'leiden_unintegrated_colors', 'log1p', 'neighbors', 'ot_graph', 'pca', 'phase_colors', 'sample_colors', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj_fates', 't

In [21]:
print('PLAU' in set(adata_thp1_hvg.var.index)) # similar shape
print('CD83' in set(adata_thp1_hvg.var.index)) # similar shape
print('EGR1' in set(adata_thp1_hvg.var.index)) # similar shape
print('IER3' in set(adata_thp1_hvg.var.index)) # similar shape

True
True
True
True


In [22]:
# TPs
print('REL' in set(adata_thp1_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_thp1_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_thp1_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_thp1_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_thp1_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_thp1_hvg.var.index)) # similar shape

True
True
True
True
True
True


In [23]:
print('HNRNPA1P27' in adata_k562_hvg.var.index)
print('HNRNPA1P27' in adata_thp1_hvg.var.index)

False
False


# Prepare input files for TradeSeq (per cell type)

In [24]:
adata_full = sc.read_h5ad('../output/adata_full.chk1.h5ad')
adata_k562_full = adata_full[adata_k562_hvg.obs.index, :]
print(adata_k562_full.shape)

adata_thp1_full = adata_full[adata_thp1_hvg.obs.index, :]
print(adata_thp1_full.shape)

(3128, 21693)
(787, 21693)


# Prepare input files for TradeSeq (across cell types)

In [25]:
adata_k562_hvg.write_h5ad('../output/adata_k562_prefiltered.fixed.h5ad')
adata_thp1_hvg.write_h5ad('../output/adata_thp1_prefiltered.fixed.h5ad')


In [26]:
cp.tl.fit_gene_dynamics(
    adata_k562_hvg,
    output_dir='../intmd/sc-K562_to_tradeSeq.fixed',
    full_adata=adata_k562_full,
    prepare_inputs=True,
    run_backend=False,
)
cp.tl.fit_gene_dynamics(
    adata_thp1_hvg,
    output_dir='../intmd/sc-THP1_to_tradeSeq.fixed',
    full_adata=adata_thp1_full,
    prepare_inputs=True,
    run_backend=False,
)


GeneDynamicsResult(backend='tradeseq', key_added='gene_dynamics', metadata={'time_key': 'time', 'layer': 'counts', 'input_dir': '../intmd/sc-THP1_to_tradeSeq.fixed', 'n_points': 9, 'workers': 4, 'screen_nknots': 4, 'final_nknots': 8, 'p_threshold': 0.05, 'outputs': {'association': 'asso_df.1trajectory.nknots4.csv', 'significant': 'sig_df.1trajectory.nknots4.csv', 'sce': 'sce.1trajectory.nknots8.rds', 'gamList': 'gamList.1trajectory.nknots8.rds'}, 'legacy_outputs': {'association': [], 'significant': [], 'sce': [], 'gamList': []}, 'prepared_paths': {'counts': '../intmd/sc-THP1_to_tradeSeq.fixed/counts_genes_x_cells.csv.gz', 'cell_times': '../intmd/sc-THP1_to_tradeSeq.fixed/cell_times.csv.gz', 'subset_genes': '../intmd/sc-THP1_to_tradeSeq.fixed/subset_genes.csv', 'full_counts': '../intmd/sc-THP1_to_tradeSeq.fixed/counts_genes_x_cells.full.csv.gz', 'full_cell_times': '../intmd/sc-THP1_to_tradeSeq.fixed/cell_times.full.csv.gz', 'full_subset_genes': '../intmd/sc-THP1_to_tradeSeq.fixed/subset

In [27]:
# Full-library tradeSeq inputs are written by the package-backed calls above.
